In [1]:
import hashlib
import os
from pathlib import Path

import your

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
def get_data_from_filterbank(filterbank_file, index, rand=0):
    steps = range(0, filterbank_file.your_header.nspectra, 256)
    return filterbank_file.get_data(nstart=steps[index]+rand, nsamp=256).T


def compute_stats(x: np.ndarray, axis=None):
    """Return mean, standard deviation, skewness, and excess kurtosis."""
    x = np.asarray(x)
    mean = x.mean(axis=axis, keepdims=True)
    std = x.std(axis=axis, keepdims=True)

    centered = x - mean
    m2 = (centered ** 2).mean(axis=axis, keepdims=True)
    m3 = (centered ** 3).mean(axis=axis, keepdims=True)
    m4 = (centered ** 4).mean(axis=axis, keepdims=True)

    std_safe = np.where(std == 0, 1.0, std)
    skew = m3 / (std_safe ** 3)
    kurt = m4 / (std_safe ** 4) - 3.0

    return mean.squeeze(), std.squeeze(), skew.squeeze(), kurt.squeeze()


def normalize_per_channel(seg: np.ndarray) -> np.ndarray:
    """Scale each channel to the range 0 to 256."""
    seg = np.asarray(seg)
    chan_min = seg.min(axis=1, keepdims=True)
    chan_max = seg.max(axis=1, keepdims=True)
    denom = chan_max - chan_min
    denom_safe = np.where(denom == 0, 1.0, denom)

    return (seg - chan_min) / denom_safe * 256.0


def compute_ratio(per_channel_vals: np.ndarray, seg_val: float) -> np.ndarray:
    """Return channel values divided by the segment value."""
    per_channel_vals = np.asarray(per_channel_vals, dtype=float)
    if seg_val == 0:
        return np.full_like(per_channel_vals, np.nan, dtype=float)
    return per_channel_vals / seg_val


def get_rfi_block_id(segment_index: int, rfi_blocks_for_file: dict):
    """Return the RFI block that contains a segment."""
    for block_id, (start, end) in rfi_blocks_for_file.items():
        if start <= segment_index <= end:
            return block_id
    return None


def get_freq_ranges_for_block(block_id: int, segments_rfi_for_file: dict):
    """Return the channel index ranges for one RFI block."""
    vals = segments_rfi_for_file[block_id]
    if isinstance(vals[0], (list, tuple, np.ndarray)):
        ranges = vals
    else:
        ranges = [vals]
    return [(int(s), int(e)) for (s, e) in ranges]


def build_rfi_mask(n_channels: int, freq_ranges):
    """Return a mask for channels inside the RFI ranges."""
    mask = np.zeros(n_channels, dtype=bool)
    for start, end in freq_ranges:
        s = max(0, start)
        e = min(n_channels - 1, end)
        if s <= e:
            mask[s:e + 1] = True
    return mask

In [3]:
filterbank_file_path = Path('/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil')
segment_labels_path = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/segments_index_B0531+21_59000_48386.csv')
output_dir = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/runs/prepublic_run')
expected_filterbank_md5 = '3cf053e6bc46911eada8d19bcb591bf5'

# Stop before processing when a required input is not available.
if not filterbank_file_path.is_file():
    raise FileNotFoundError(f'Filterbank file not found: {filterbank_file_path}')
if not segment_labels_path.is_file():
    raise FileNotFoundError(f'Segment labels file not found: {segment_labels_path}')
if not output_dir.is_dir():
    raise FileNotFoundError(f'Output directory not found: {output_dir}')

def md5sum(path, block_size=1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

file_hash = md5sum(filterbank_file_path)
if file_hash != expected_filterbank_md5:
    raise ValueError(f'Unexpected filterbank MD5: {file_hash}')



frequency_range = {
    'B0531+21_59000_48386.fil': np.linspace(1530.0, 1210.0, 256)
}

rfi_blocks = {
    'B0531+21_59000_48386.fil': {
        1: (204,1527,),
        2: (2023, 4436,),
        3: (4437, 4953,),
        4: (6701, 7631,),
        5: (42546, 45015,),
        6: (45622, 45705,),
        7: (46186, 46927,),
        8: (46928, 47303,),
        9: (48686, 48831,),
        10: (48832, 49591,)
    }
}

segments_rfi_indexes = {
    'B0531+21_59000_48386.fil': {
        1: [[230, 235], [239, 241]],
        2: [230, 256],
        3: [230, 256],
        4: [[230,235], [239, 243]],
        5: [[189, 195], [199, 202], [205, 209]],
        6: [[189, 195], [199, 202], [206, 210]],
        7: [[190, 193], [199, 202], [206, 210]],
        8: [[188, 194], [198, 203], [206, 210]],
        9: [[188, 194], [198, 203], [206, 210]],
        10: [[188, 194], [198, 203], [206, 210]]
    }
}

In [4]:
filterbank_file = your.Your(str(filterbank_file_path))
labels = pd.read_csv(segment_labels_path)

required_columns = {'segment_index', 'global_index', 'label'}
missing_columns = required_columns.difference(labels.columns)
if missing_columns:
    raise ValueError(f'Missing annotation columns: {sorted(missing_columns)}')

In [5]:
filterbank_file.your_header

Unified Header:
  "basename": "B0531+21_59000_48386"
  "bw": -320.0
  "center_freq": 1370.625
  "dec_deg": null
  "dtype": "uint8"
  "fch1": 1530.0
  "filelist": [
    "/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil"
  ]
  "filename": "/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil"
  "foff": -1.25
  "format": "fil"
  "frequency_decimation_factor": 1
  "gb": null
  "gl": null
  "native_foff": -1.25
  "native_nbits": 8
  "native_nchans": 256
  "native_nspectra": 12818359.0
  "native_tsamp": 0.0001024
  "nbits": 8
  "nchans": 256
  "npol": 1
  "nspectra": 12818359
  "poln_order": "I"
  "ra_deg": null
  "source_name": "B1937+21"
  "time_decimation_factor": 1
  "tsamp": 0.0001024
  "tstart": 59000.48386574074
  "tstart_utc": "2020-05-31T11:36:46.000"

In [6]:
file_key = filterbank_file_path.name
freqs = frequency_range[file_key]
rfi_blocks_file = rfi_blocks[file_key]
segments_rfi_file = segments_rfi_indexes[file_key]

all_channels = []
meta_rows = []

# These labels define the reported training universe.
interesting = labels[labels['label'].isna() |
                     (labels['label'] == 'bright_NBRFI')]

n_time_default = 256

for _, row in tqdm(interesting.iterrows(),
                   total=len(interesting),
                   desc='Processing segments'):
    seg_idx = int(row['segment_index'])
    glob_idx = int(row['global_index'])
    orig_label = row['label']

    if pd.isna(orig_label):
        segment_type = 'nan'
        rfi_block_id = None
    elif orig_label == 'bright_NBRFI':
        segment_type = 'bright_NBRFI'
        rfi_block_id = get_rfi_block_id(seg_idx, rfi_blocks_file)
        # The annotations define RFI channels only inside these blocks.
        if rfi_block_id is None:
            continue
    else:
        continue

    seg = get_data_from_filterbank(filterbank_file, seg_idx, rand=0)
    seg = np.asarray(seg)
    n_channels, n_time = seg.shape
    n_time_default = n_time

    mean_o_seg, std_o_seg, skew_o_seg, kurt_o_seg = compute_stats(seg)

    seg_norm = normalize_per_channel(seg)

    mean_n_seg, std_n_seg, skew_n_seg, kurt_n_seg = compute_stats(seg_norm)

    mean_o_chan, std_o_chan, skew_o_chan, kurt_o_chan = compute_stats(seg, axis=1)
    mean_n_chan, std_n_chan, skew_n_chan, kurt_n_chan = compute_stats(seg_norm, axis=1)

    mean_o_ratio = compute_ratio(mean_o_chan, mean_o_seg)
    std_o_ratio  = compute_ratio(std_o_chan, std_o_seg)
    skew_o_ratio = compute_ratio(skew_o_chan, skew_o_seg)
    kurt_o_ratio = compute_ratio(kurt_o_chan, kurt_o_seg)

    mean_n_ratio = compute_ratio(mean_n_chan, mean_n_seg)
    std_n_ratio  = compute_ratio(std_n_chan, std_n_seg)
    skew_n_ratio = compute_ratio(skew_n_chan, skew_n_seg)
    kurt_n_ratio = compute_ratio(kurt_n_chan, kurt_n_seg)

    if segment_type == 'nan':
        # Keep clean segments outside the RFI class.
        channel_labels = [np.nan] * n_channels
    else:
        freq_ranges = get_freq_ranges_for_block(rfi_block_id, segments_rfi_file)
        rfi_mask = build_rfi_mask(n_channels, freq_ranges)
        channel_labels = ['NBRFI' if is_rfi else 'NoneWNBRFI'
                          for is_rfi in rfi_mask]

    for ch_idx in range(n_channels):
        sample_index = len(all_channels)

        ch_series = seg[ch_idx]
        all_channels.append(ch_series.astype(np.float32))

        meta_rows.append({
            'sample_index': sample_index,
            'global_index': glob_idx,
            'segment_index': seg_idx,
            'channel_index': ch_idx,
            'frequency': float(freqs[ch_idx]),

            'mean_o': float(mean_o_chan[ch_idx]),
            'std_o': float(std_o_chan[ch_idx]),
            'skew_o': float(skew_o_chan[ch_idx]),
            'kurt_o': float(kurt_o_chan[ch_idx]),

            'mean_n': float(mean_n_chan[ch_idx]),
            'std_n': float(std_n_chan[ch_idx]),
            'skew_n': float(skew_n_chan[ch_idx]),
            'kurt_n': float(kurt_n_chan[ch_idx]),

            'mean_o_ratio': float(mean_o_ratio[ch_idx]),
            'std_o_ratio': float(std_o_ratio[ch_idx]),
            'skew_o_ratio': float(skew_o_ratio[ch_idx]),
            'kurt_o_ratio': float(kurt_o_ratio[ch_idx]),

            'mean_n_ratio': float(mean_n_ratio[ch_idx]),
            'std_n_ratio': float(std_n_ratio[ch_idx]),
            'skew_n_ratio': float(skew_n_ratio[ch_idx]),
            'kurt_n_ratio': float(kurt_n_ratio[ch_idx]),

            'label': channel_labels[ch_idx],
            'original_segment_label': orig_label
        })

if all_channels:
    channels_array = np.stack(all_channels, axis=0)
else:
    channels_array = np.empty((0, n_time_default), dtype=np.float32)

meta_df = pd.DataFrame(meta_rows)

if len(meta_df) != len(channels_array):
    raise RuntimeError('Metadata rows do not match channel profiles')
if not np.array_equal(meta_df['sample_index'].to_numpy(), np.arange(len(meta_df))):
    raise RuntimeError('Metadata sample indices do not match channel rows')

base_name = filterbank_file_path.stem
channels_npy_path = output_dir / f'{base_name}_channels.npy'
meta_csv_path = output_dir / f'{base_name}_channels_meta.csv'

np.save(channels_npy_path, channels_array)
meta_df.to_csv(meta_csv_path, index=False)

print('Complete')
print('Channel array:', channels_array.shape)
print('Metadata rows:', len(meta_df))
print('Output files:')
print('  ', channels_npy_path)
print('  ', meta_csv_path)

Processing segments: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26656/26656 [21:06<00:00, 21.04it/s]


Complete
Channel array: (6823936, 256)
Metadata rows: 6823936
Output files:
   /u/akazantsev/ml_dl_for_rfi_mitigation/runs/prepublic_run/B0531+21_59000_48386_channels.npy
   /u/akazantsev/ml_dl_for_rfi_mitigation/runs/prepublic_run/B0531+21_59000_48386_channels_meta.csv
